In [3]:
from pyspark.sql import SparkSession
import pandas as pd
from pymongo import MongoClient

print("Iniciando PySpark para escribir en HDFS...")

spark = SparkSession.builder \
    .appName("Spark_to_HDFS") \
    .getOrCreate()

try:
    # 1. Leer de Mongo
    client = MongoClient('mongodb://mongodb:27017')
    db = client['steam_db']
    datos = list(db.reviews_raw.find({}, {"_id":0, "language":1, "voted_up":1, "playtime_hours":1}))
    df_pandas = pd.DataFrame(datos)

    # 2. Spark DataFrame
    df_spark = spark.createDataFrame(df_pandas)

    # 3. Procesamiento Big Data
    analisis = df_spark.groupBy("language") \
                       .agg({"playtime_hours": "avg", "language": "count"}) \
                       .withColumnRenamed("avg(playtime_hours)", "Media_Horas") \
                       .withColumnRenamed("count(language)", "Total_Resenas")

    # 4. Guardar en HDFS
    # hdfs://namenode:8020 es la nueva dirección del clúster estable
    ruta_hdfs = "hdfs://namenode:8020/user/datos_spark/analisis_idiomas.csv"
    
    print(f"Escribiendo en Hadoop HDFS: {ruta_hdfs} ...")
    
    # Escribimos el CSV
    analisis.coalesce(1).write.mode("overwrite").option("header", "true").csv(ruta_hdfs)
    
    print("Datos guardados en el Datanode de HDFS.")

except Exception as e:
    print(f"Error: {e}")
finally:
    spark.stop()

Iniciando PySpark para escribir en HDFS...
Escribiendo en Hadoop HDFS: hdfs://namenode:8020/user/datos_spark/analisis_idiomas.csv ...
Datos guardados en el Datanode de HDFS.
